# Chapter 19 — Volatility Smiles and Surfaces

The chapter's smiles, densities and surfaces, built from their definitions and made
to reproduce the chapter's numbers. **NumPy only** (matplotlib for the plots). The
Breeden–Litzenberger density, the variance strip, the wing bound, the sticky-delta
correction and the surface interpolation are written out rather than called from a
library, so the formula in the text and the code are the same object.

| Section | What is built | Text |
|---|---|---|
| 1 | Black–Scholes–Merton, vega, implied volatility, a `show` helper | — |
| 2 | The skew, its slope, and standardized moneyness | Figure 19.1, Table 19.1 |
| 3 | The density a smile implies, and its tails | (19.1), Example 19.1, Figure 19.2 |
| 4 | A distribution from six quotes | Example 19.2 |
| 5 | The variance strip on the one-year skew | (19.2), §19.3 |
| 6 | How fast a wing may steepen | Example 19.3 |
| 7 | The variance of a scheduled event | §19.6 |
| 8 | Implied correlation | (19.3), Example 19.4 |
| 9 | Sticky strike, sticky delta and the delta correction | (19.4), Example 19.5, Figure 19.4 |
| 10 | The S&P 500 surface, and pricing an option that does not trade | Table 19.3, Example 19.6 |

Every printed figure is shown beside the number the chapter prints, with `ok` when
they agree to the chapter's rounding. Figures the chapter does not print are printed
without a book value.

## 1. The formulas and a checking helper

`bs` is the Black–Scholes–Merton price with a yield $q$, `vega` its sensitivity to
volatility, and `implied_vol` inverts a price by bisection, which works because the
price rises with volatility. `show` prints a computed value beside the chapter's.

In [ ]:
import numpy as np
from math import log, sqrt, exp, erf, pi


def N(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


def phi(x):
    return exp(-0.5 * x * x) / sqrt(2.0 * pi)


def show(label, got, printed, tol=None):
    # tol defaults to half a unit in the last printed digit
    if tol is None and isinstance(printed, int):
        tol = 0.5
    elif tol is None:
        s = repr(float(printed)).split(".")
        tol = 0.5 * 10 ** -len(s[1]) + 1e-12
    flag = "ok" if abs(got - printed) <= tol else "DIFFERS"
    print("%-56s %12.6g   book %10.6g   %s" % (label, got, printed, flag))


def note(label, got):
    print("%-56s %12.6g" % (label, got))


def d12(S, K, T, r, sig, q=0.0):
    d1 = (log(S / K) + (r - q + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    return d1, d1 - sig * sqrt(T)


def bs(S, K, T, r, sig, q=0.0, call=True):
    d1, d2 = d12(S, K, T, r, sig, q)
    c = S * exp(-q * T) * N(d1) - K * exp(-r * T) * N(d2)
    return c if call else c - S * exp(-q * T) + K * exp(-r * T)


def vega(S, K, T, r, sig, q=0.0):
    d1, _ = d12(S, K, T, r, sig, q)
    return S * exp(-q * T) * phi(d1) * sqrt(T)


def implied_vol(price, S, K, T, r, q=0.0, call=True):
    lo, hi = 1e-4, 3.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if bs(S, K, T, r, mid, q, call) > price:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)

## 2. The skew — Figure 19.1 and Table 19.1

Figure 19.1 is drawn from $\sigma = 0.20 - 0.055z + 0.018z^2$ with
$z = \ln(K/S)/(0.20\sqrt{T})$. The first lines confirm §19.1's opening point: a call
and a put at the same strike, priced at the same volatility, invert to that one
volatility, because parity ties them together. Then the slope between the 90 and 110
strikes, per 10 points of strike, falls from 5.7 to 1.6 as maturity lengthens, while
the same curves measured per standard deviation of moneyness are nearly one curve.

In [ ]:
def skew(K, S=100.0, T=1.0):
    # Figure 19.1's smile
    z = log(K / S) / (0.20 * sqrt(T))
    return 0.20 - 0.055 * z + 0.018 * z * z


s90 = skew(90.0)
c90 = bs(100.0, 90.0, 1.0, 0.03, s90)
p90 = bs(100.0, 90.0, 1.0, 0.03, s90, call=False)
note("implied vol from the 90 call (r = 3%)", implied_vol(c90, 100.0, 90.0, 1.0, 0.03))
note("implied vol from the 90 put", implied_vol(p90, 100.0, 90.0, 1.0, 0.03, call=False))
print()
for T, printed in ((0.25, 5.7), (1.0, 2.8), (3.0, 1.6)):
    show("slope at T = %.2f, points per 10 of strike" % T,
         100 * (skew(90.0, T=T) - skew(110.0, T=T)) / 2, printed)
print()
for T, printed in ((0.25, 1.05), (1.0, 0.53), (3.0, 0.30)):
    show("the 90 strike in standard deviations, T = %.2f" % T, -log(0.9) / (0.20 * sqrt(T)), printed)
print()
for T, printed in ((0.25, 5.68), (1.0, 5.59), (3.0, 5.56)):
    lo, hi = round(100 * skew(90.0, T=T), 2), round(100 * skew(110.0, T=T), 2)   # the plotted points
    dz = (log(1.1) - log(0.9)) / (0.20 * sqrt(T))
    show("slope per standard deviation, T = %.2f" % T, (lo - hi) / dz, printed)

try:
    import matplotlib.pyplot as plt
    Ks = np.linspace(70, 130, 121)
    plt.figure(figsize=(6, 3.6))
    for T, style, lab in ((0.25, "-", "3 months"), (1.0, "--", "1 year"), (3.0, ":", "3 years")):
        plt.plot(Ks, [100 * skew(k, T=T) for k in Ks], style, label=lab)
    plt.xlabel("strike")
    plt.ylabel("implied volatility (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 3. The density a smile implies — equation (19.1) and Example 19.1

Breeden–Litzenberger: $g(K) = e^{rT}\,\partial^2 c/\partial K^2$, taken along the
smile. Example 19.1 uses the milder one-year smile
$\sigma(K) = 0.20 - 0.15\ln(K/100) + 0.15\ln^2(K/100)$ with $r = 0$. Each tail
probability is a digital, $\Pr(S_T < K) = 1 + dc/dK$, and $dc/dK$ is a total
derivative along the smile: the flat-volatility digital plus vega times the smile's
slope. The control comes first: on a flat 20% smile the same code must return the
lognormal.

In [ ]:
S0, T1 = 100.0, 1.0


def smile1(K):
    x = log(K / S0)
    return 0.20 - 0.15 * x + 0.15 * x * x


def dsmile1(K):
    return (-0.15 + 0.30 * log(K / S0)) / K


def prob_below(K, sig, dsig):
    # 1 + dc/dK with r = 0: the digital put along the smile
    s = sig(K)
    _, d2 = d12(S0, K, T1, 0.0, s)
    return N(-d2) + vega(S0, K, T1, 0.0, s) * dsig(K)


def density(K, sig):
    # equation (19.1) by central difference
    h = 1e-4 * K
    c = lambda k: bs(S0, k, T1, 0.0, sig(k))
    return (c(K - h) - 2 * c(K) + c(K + h)) / (h * h)


flat = lambda K: 0.20
dflat = lambda K: 0.0
show("control: flat smile, Pr(S_T < 70) against lognormal",
     prob_below(70, flat, dflat), N((log(0.70) + 0.02) / 0.20), tol=1e-12)
print()
p = {}
for K, left, ps, pl in ((60, True, 3.0, 0.7), (70, True, 6.9, 4.6), (80, True, 14.8, 15.5), (130, False, 6.0, 7.9)):
    a, b = prob_below(K, smile1, dsmile1), prob_below(K, flat, dflat)
    if not left:
        a, b = 1 - a, 1 - b
    p[K] = (100 * a, 100 * b)
    side = "<" if left else ">"
    show("Pr(S_T %s %d), smile (%%)" % (side, K), 100 * a, ps)
    show("Pr(S_T %s %d), lognormal (%%)" % (side, K), 100 * b, pl)
print()
show("ratio at 60 (more than four times)", p[60][0] / p[60][1], 4.0, tol=0.5)
show("ratio at 70 (1.5 times)", p[70][0] / p[70][1], 1.5)

The caveat of Example 19.1: extended as a quadratic, the smile's density turns
negative far below the quoted strikes, near 6, and the density integrated upward from
there totals 1.06 instead of 1. The cell also finds where the two densities cross,
the 65-to-97 band in which the market's density runs below the lognormal, and draws
Figure 19.2.

In [ ]:
def root(f, a, b):
    fa = f(a)
    for _ in range(100):
        m = 0.5 * (a + b)
        fm = f(m)
        if (fa < 0) == (fm < 0):
            a, fa = m, fm
        else:
            b = m
    return 0.5 * (a + b)


def lognormal_density(K):
    m, s = log(S0) - 0.02, 0.20
    return phi((log(K) - m) / s) / (K * s)


g = lambda K: density(K, smile1)
k_neg = root(g, 3.0, 10.0)
show("density turns negative below (about 6)", k_neg, 6, tol=0.5)
show("density integrated from there upward", 1 - prob_below(k_neg, smile1, dsmile1), 1.06)
diff = lambda K: g(K) - lognormal_density(K)
show("densities cross, left shoulder starts", root(diff, 60, 80), 65, tol=0.5)
show("densities cross, left shoulder ends", root(diff, 85, 100), 97, tol=0.5)

grid = np.linspace(10.0, 400.0, 39001)
vals = np.array([g(k) for k in grid])
mass = float(np.sum(0.5 * (vals[1:] + vals[:-1]) * np.diff(grid)))
note("density integrated from 10 to 400 (over 1: Pr(S_T < 10) < 0)", mass)

try:
    import matplotlib.pyplot as plt
    Ks = np.linspace(40, 172, 133)
    plt.figure(figsize=(6, 3.6))
    plt.plot(Ks, [g(k) for k in Ks], label="implied by the smile")
    plt.plot(Ks, [lognormal_density(k) for k in Ks], "--", label="lognormal, 20%")
    plt.xlabel("S_T")
    plt.ylabel("risk-neutral density")
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 4. A distribution from six quotes — Example 19.2

The chain of Table 10.2: six calls \$2.50 apart, $r = 5\%$, $T = 45/365$. Each
butterfly is a second difference, each outer call spread a digital, and the three
pieces sum to 100% by telescoping whatever the quotes are. The lognormals for
comparison are centered on Example 11.4's implied forward.

In [ ]:
T, r, h = 45 / 365, 0.05, 2.50
K = [42.50, 45.00, 47.50, 50.00, 52.50, 55.00]
C = [6.650, 4.425, 2.670, 1.475, 0.710, 0.305]
grow = exp(r * T)

fly = [C[i - 1] - 2 * C[i] + C[i + 1] for i in range(1, 5)]
dens = [grow * b / h**2 for b in fly]
for k, b, gk, pb, pg in zip(K[1:5], fly, dens, (0.4700, 0.5600, 0.4300, 0.3600), (0.0757, 0.0902, 0.0692, 0.0580)):
    show("butterfly at %.2f" % k, b, pb)
    show("density g(K) at %.2f" % k, gk, pg)
print()
bucket = [100 * gk * h for gk in dens]
for k, b, pb in zip(K[1:5], bucket, (18.9, 22.5, 17.3, 14.5)):
    show("bucket around %.2f (%%)" % k, b, pb)
centre = sum(bucket)
show("probability from 43.75 to 53.75 (%)", centre, 73.3)
up_lo = 100 * grow * (C[0] - C[1]) / h
up_hi = 100 * grow * (C[4] - C[5]) / h
show("Pr(S_T > 43.75) (%)", up_lo, 89.6)
show("Pr(S_T > 53.75) (%)", up_hi, 16.3)
show("Pr(S_T < 43.75) (%)", 100 - up_lo, 10.4)
show("the three pieces (an identity) (%)", 100 - up_lo + centre + up_hi, 100.0, tol=1e-9)
print()
F_impl = 47.50 + (2.67 - 2.575) * exp(r * T)   # Example 11.4's implied forward
show("Example 11.4's forward", F_impl, 47.60)


def band(s, lo=43.75, hi=53.75):
    m, sd = log(F_impl) - 0.5 * s * s * T, s * sqrt(T)
    return N((log(hi) - m) / sd) - N((log(lo) - m) / sd)


show("lognormal at 39.7%, same band (%)", 100 * band(0.397), 53.0)
lo, hi = 0.05, 0.60
for _ in range(200):
    mid = 0.5 * (lo + hi)
    lo, hi = (mid, hi) if band(mid) > centre / 100 else (lo, mid)
show("lognormal volatility that puts 73.3% there (%)", 100 * mid, 25.6)
show("each $0.10 of early-exercise premium, tail points", 100 * grow * 0.10 / h, 4.0)

## 5. The variance strip on the one-year skew — equation (19.2) and §19.3

The $1/K^2$ weights first: a strike at 60 carries 2.8 times the at-the-money weight,
one at 140 about half. Then equation (19.2) on Figure 19.1's one-year coordinates,
interpolated linearly and held flat beyond 70 and 130. The flat-smile control must
return 20.00% over a wide range; truncated to 70–130 even the flat smile loses half a
point. The skewed strip comes out more than four points above the at-the-money 20%,
the figure Section 22.6 measures.

In [ ]:
for k, printed in ((60, 2.8), (140, 0.5)):
    show("1/K^2 weight at %d, relative to 100" % k, (100 / k) ** 2, printed, tol=0.05)

fig_K = np.array([70, 75, 80, 85, 90, 95, 100, 105, 110, 115, 120, 125, 130], float)
fig_v = np.array([35.53, 31.64, 28.38, 25.66, 23.40, 21.53, 20.00, 18.77, 17.79, 17.04, 16.48, 16.10, 15.88])
Nv = np.vectorize(N)


def var_strike(lo, hi, flat=None, steps=40000, S=100.0, T=1.0, r=0.0):
    # equation (19.2) by the midpoint rule; out-of-the-money options on each side of F
    F = S * exp(r * T)
    w = (hi - lo) / steps
    k = lo + (np.arange(steps) + 0.5) * w
    s = np.full_like(k, flat) if flat is not None else np.interp(k, fig_K, fig_v) / 100
    d1 = (np.log(S / k) + (r + 0.5 * s * s) * T) / (s * sqrt(T))
    c = S * Nv(d1) - k * exp(-r * T) * Nv(d1 - s * sqrt(T))
    opt = np.where(k < F, c - S + k * exp(-r * T), c)
    return 100 * sqrt(2 * exp(r * T) / T * np.sum(w * opt / k**2))


show("control: flat 20%, strikes 50-200", var_strike(50, 200, flat=0.20), 20.00)
show("flat 20%, truncated to 70-130", var_strike(70, 130, flat=0.20), 19.56)
show("one-year skew, strikes 50-200", var_strike(50, 200), 24.38)
show("one-year skew, truncated to 70-130", var_strike(70, 130), 21.99)

## 6. How fast a wing may steepen — Example 19.3

A call spread turns negative once $d\sigma/dK > e^{-rT}N(d_2)/\mathcal{V}$. The table
gives that ceiling per 5 points of strike; the chain that breaches it produces a
115/120 call spread priced at −\$0.3878, and bisection finds the largest
$\sigma(120)$ the chain could have carried.

In [ ]:
S, T, r = 100.0, 0.5, 0.03
for k, s, pv, pcap in ((110, 0.225, 25.77, 5.34), (115, 0.240, 22.83, 4.45),
                       (120, 0.260, 20.19, 3.86), (125, 0.290, 18.60, 3.49)):
    v = vega(S, k, T, r, s)
    _, d2 = d12(S, k, T, r, s)
    show("vega at %d" % k, v, pv)
    show("maximum steepening per 5 of strike at %d (points)" % k, 5 * exp(-r * T) * N(d2) / v * 100, pcap)
print()
chain = ((100, 0.240, 7.4824), (105, 0.232, 5.0715), (110, 0.228, 3.3237), (115, 0.250, 2.6563), (120, 0.310, 3.0442))
prices = []
for k, s, pc in chain:
    prices.append(bs(S, k, T, r, s))
    show("call at %d, sigma %.1f%%" % (k, 100 * s), prices[-1], pc)
for (k, _, _), a, b, pspread in zip(chain[1:], prices, prices[1:], (2.4109, 1.7478, 0.6673, -0.3878)):
    show("call spread ending at %d" % k, a - b, pspread)
show("implied digital, 115/120", exp(r * T) * (prices[3] - prices[4]) / 5, -0.0787)
s_max = implied_vol(prices[3], S, 120, T, r)
show("largest sigma(120) the chain tolerates (%)", 100 * s_max, 29.27)
show("the quoted 31.0% past it (points)", 31.0 - 100 * s_max, 1.7)

## 7. The variance of a scheduled event — §19.6

Total variance is additive in time. Fifteen trading days at 35% contain fourteen
ordinary days at 25% and the event day, so the event day's variance is what is left.

In [ ]:
ev = (0.35**2 * 15 - 0.25**2 * 14) / 252
show("event-day variance", ev, 0.0038)
show("event-day standard deviation (%)", 100 * sqrt(ev), 6.2)
show("ordinary-day standard deviation (%)", 100 * 0.25 / sqrt(252), 1.6)

## 8. Implied correlation — equation (19.3) and Example 19.4

With $n$ equal weights and one correlation, (19.3) gives
$\sigma_I^2 = \sigma^2[1/n + \rho(1 - 1/n)]$, which solves for $\rho$. The cell then
prices the index at 25% realized correlation; the gap to 18% is what the dispersion
trade collects. The plot shows index volatility across correlations.

In [ ]:
n, s_comp, s_idx = 50, 0.30, 0.18
rho_impl = ((s_idx / s_comp) ** 2 - 1 / n) / (1 - 1 / n)
show("implied correlation (%)", 100 * rho_impl, 34.7)
idx25 = s_comp * sqrt(1 / n + 0.25 * (1 - 1 / n))
show("index volatility at 25% correlation (%)", 100 * idx25, 15.4)
show("the gap dispersion collects (points)", 100 * (s_idx - idx25), 2.6)

try:
    import matplotlib.pyplot as plt
    rhos = np.linspace(0, 1, 101)
    plt.figure(figsize=(6, 3.6))
    plt.plot(100 * rhos, 100 * s_comp * np.sqrt(1 / n + rhos * (1 - 1 / n)), label="index volatility")
    plt.axhline(100 * s_idx, color="k", ls="--", lw=0.8, label="index implied, 18%")
    plt.axvline(100 * rho_impl, color="k", ls=":", lw=0.8)
    plt.xlabel("common correlation (%)")
    plt.ylabel("volatility (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 9. Sticky delta and the delta correction — equation (19.4), Example 19.5, Figure 19.4

Under sticky delta the smile is a function of $K/S$, so a rise in spot hands each
strike a new volatility. Equation (19.4) adds vega times $\partial\sigma/\partial S$
to the Black–Scholes delta; $\partial\sigma/\partial S$ is taken by central difference
on the one-year skew of Figure 19.1, $r = 0$. Figure 19.4's sticky-delta curve after
the move to 105 is $\sigma(K)$ evaluated with spot 105.

In [ ]:
S = 100.0


def dsig_dS(k, spot, eps=1e-4):
    return (skew(k, spot + eps) - skew(k, spot - eps)) / (2 * eps)


rows = ((90, False, 0.2340, -0.2853, 0.1256, -0.1596),
        (100, True, 0.2000, 0.5398, 0.1092, 0.6490),
        (110, True, 0.1779, 0.3275, 0.0683, 0.3958))
for k, is_call, ps, pd, pc, pt in rows:
    s = skew(k, S)
    d1, _ = d12(S, k, 1.0, 0.0, s)
    delta = N(d1) - (0.0 if is_call else 1.0)
    corr = vega(S, k, 1.0, 0.0, s) * dsig_dS(k, S)
    kind = "call" if is_call else "put"
    show("sigma at %d" % k, s, ps)
    show("Black-Scholes delta, %d %s" % (k, kind), delta, pd)
    show("correction, %d %s" % (k, kind), corr, pc)
    show("total delta, %d %s" % (k, kind), delta + corr, pt, tol=1e-4)
atm_corr = vega(S, 100, 1.0, 0.0, 0.20) * dsig_dS(100, S)
show("shares on 100,000 at-the-money options", 100000 * atm_corr, 10920, tol=5)
print()
for k, printed in ((90, 25.31), (95, 23.20), (100, 21.45), (105, 20.00), (110, 18.82), (120, 17.13)):
    show("sticky-delta smile after the move to 105, K = %d (%%)" % k, 100 * skew(k, 105.0), printed)

try:
    import matplotlib.pyplot as plt
    Ks = np.linspace(85, 125, 81)
    plt.figure(figsize=(6, 3.6))
    plt.plot(Ks, [100 * skew(k, 100.0) for k in Ks], label="before, and sticky strike after")
    plt.plot(Ks, [100 * skew(k, 105.0) for k in Ks], "--", label="sticky delta after the move to 105")
    plt.scatter([100, 100], [20.00, 100 * skew(100, 105.0)], color="k", zorder=3)
    plt.xlabel("strike")
    plt.ylabel("implied volatility (%)")
    plt.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

## 10. The S&P 500 surface — Table 19.3 and Example 19.6

The cell reads the skew off each row, standardizes it by the row's own at-the-money
volatility, and checks §19.5's calendar condition: total variance must rise down every
column. Example 19.6 then prices a nine-month put at 92% of spot: linear across the
strikes, then linear in total variance down the maturities. Priced at the
at-the-money volatility instead, the same put loses the skew.

In [ ]:
strikes = [80, 90, 95, 100, 105, 110, 120]
years = [1 / 12, 0.25, 0.5, 1.0, 2.0]
surface = np.array([[30.0, 22.4, 19.6, 17.5, 15.8, 14.9, 15.6],
                    [27.5, 21.9, 19.9, 18.5, 17.1, 16.0, 14.8],
                    [26.0, 21.7, 20.3, 19.3, 18.2, 17.2, 15.6],
                    [25.0, 21.9, 20.8, 20.0, 19.3, 18.5, 17.2],
                    [24.2, 22.0, 21.3, 20.8, 20.2, 19.7, 18.7]])
col = {k: surface[:, i] for i, k in enumerate(strikes)}

slope = (col[90] - col[110]) / 2
for i, printed in ((0, 3.75), (3, 1.70), (4, 1.15)):
    show("skew 90-110 at %.3g years, points per 10%% of spot" % years[i], slope[i], printed)
zslope = [(col[90][i] - col[110][i]) * col[100][i] / 100 * sqrt(t) / (log(1.1) - log(0.9))
          for i, t in enumerate(years)]
show("skew per standard deviation, one month", zslope[0], 1.89)
show("skew per standard deviation, two years", zslope[4], 3.37)
w = (surface / 100) ** 2 * np.array(years)[:, None]
note("total variance rises down every column (1 = yes)", float(np.all(np.diff(w, axis=0) > 0)))
show("total variance, 110 strike, one month", w[0, 5], 0.0019)
show("total variance, 110 strike, two years", w[4, 5], 0.0776)
print()
f = (92 - 90) / (95 - 90)
s6 = col[90][2] + f * (col[95][2] - col[90][2])
s12 = col[90][3] + f * (col[95][3] - col[90][3])
show("six-month volatility at 92 (%)", s6, 21.14)
show("one-year volatility at 92 (%)", s12, 21.46)
w6, w12 = (s6 / 100) ** 2 * 0.5, (s12 / 100) ** 2
w9 = w6 + 0.5 * (w12 - w6)
show("total variance at six months", w6, 0.02234)
show("total variance at one year", w12, 0.04605)
show("total variance at nine months", w9, 0.03420)
sig9 = 100 * sqrt(w9 / 0.75)
show("nine-month volatility at 92 (%)", sig9, 21.35)
show("interpolating the volatilities instead (%)", (s6 + s12) / 2, 21.30)
wa = ((col[100][2] / 100) ** 2 * 0.5 + (col[100][3] / 100) ** 2) / 2
atm9 = 100 * sqrt(wa / 0.75)
show("nine-month at-the-money volatility (%)", atm9, 19.77)
quoted = bs(5000, 4600, 0.75, 0.04, sig9 / 100, q=0.015, call=False)
naive = bs(5000, 4600, 0.75, 0.04, atm9 / 100, q=0.015, call=False)
show("the put off the surface (index points)", quoted, 157.9)
show("the put at the at-the-money volatility", naive, 136.2)
show("the gap (index points)", quoted - naive, 21.7)
show("the gap, % of the correct price", 100 * (quoted - naive) / quoted, 13.8)

## What this notebook shows

* One volatility per strike: parity makes a call and a put invert to the same number.
* A smile is a distribution. Its second strike derivative is the density, its first is
  the digital, and an equity skew is a fat left tail and a thin right one in
  Black–Scholes units.
* Six quotes give a tail's mass but not its shape; the 100% total is an identity.
* The $1/K^2$ strip turns the skew into a variance strike above at-the-money volatility,
  and truncation alone costs something even on a flat smile.
* A wing has a hard slope ceiling, and a chain can breach it by less than two
  volatility points while looking smooth.
* Sticky delta changes every hedge: the correction is vega times the smile's drift
  with the spot.
* An option that does not trade is priced off the surface, interpolating in total
  variance down the maturities; priced off the at-the-money volatility it loses the skew.

Every figure above is deterministic and reproduces the chapter.